In [1]:
import os
import pdal
import numpy as np
import json
import laspy
from pyproj import CRS
from config import *
import rasterio

In [2]:
data_dir = '/home/ajai-krishna/work/GEO_AI/data/'
out_dir = '/home/ajai-krishna/work/GEO_AI/outputs/'

In [3]:

crs = CRS.from_proj4("+proj=utm +zone=43 +datum=WGS84 +units=m +no_defs")

In [4]:
gujarath_data_path = '/home/ajai-krishna/work/GEO_AI/data/input/laz/DEVDI_POINT CLOUD (511671).las'
gujarath_las = laspy.open(gujarath_data_path)

In [ ]:
def extract_dtm_from_las(las_file, output_path,outfile_name=''):
    pdal_pipeline = {
        "pipeline": [
            {
                "type": "readers.las",
                "filename": str(las_file)  # explicitly cast to str, e.g. if it's a Path object
            },
            {
                "type": "filters.smrf"
            },
            {
                "type": "filters.range",
                "limits": "Classification[2:2]"
            },
            # ensure the RGB dimensions are preserved by filtering on their full range
            {
                "type": "filters.range",
                "limits": "Red[0:65535],Green[0:65535],Blue[0:65535]"
            },
            {
              "type": "writers.gdal",
              "filename": output_path,
              "resolution": 0.09,
              "output_type": "min",
              "data_type": "float32"
            },
            {"type":"writers.gdal",
             "filename":f"{outfile_name}_dtm.tif",
             "resolution":0.09,
             "output_type":"min",
             "data_type":"float32"

            }      
            
        ]
    }
    
    

In [12]:
pdal_pipeline


{'pipeline': [{'type': 'readers.las',
   'filename': '/home/ajai-krishna/work/GEO_AI/data/input/laz/DEVDI_POINT CLOUD (511671).las'},
  {'type': 'filters.smrf'},
  {'type': 'filters.range', 'limits': 'Classification[2:2]'},
  {'type': 'writers.gdal',
   'filename': 'output_dem.tif',
   'resolution': 0.09,
   'output_type': 'min',
   'data_type': 'float32'}]}

In [ ]:
output_tif = os.path.join(output_dir,"ground_points.tif")
pdal_pipeline['pipeline'][-1]['filename'] = output_tif
pipeline = pdal.Pipeline(json.dumps(pdal_pipeline))
pipeline.execute()
arrays = pipeline.arrays
ground_points = arrays[0]

print(f"Ground point count: {len(ground_points)}")
print(f"Fields available: {ground_points.dtype.names}")

x = ground_points["X"]
y = ground_points["Y"]
z = ground_points["Z"]


In [ ]:
header = laspy.LasHeader(point_format=1, version="1.2")
las = laspy.LasData(header=header)
header.add_crs(crs)

las.x = ground_points["X"]
las.y = ground_points["Y"]
las.z = ground_points["Z"]


# las.write(output_gpkg)

In [ ]:
input_crs_wkt = laspy.read(gujarath_data[0]).header.parse_crs().to_wkt()
with rasterio.open(output_tif,"r+") as dtm_file:
    dtm_file.crs = rasterio.crs.CRS.from_wkt(input_crs_wkt)

In [ ]:
output_tif

In [ ]:
points = np.column_stack((las.x, las.y, las.z))
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

In [ ]:
# o3d.visualization.draw_geometries([pcd])